# Curanex: NHANES lab round C - low kidney filtering, protein in urine, hepatitis C (CPU)

**Rules fixed before running (2026-11-12):** truth measured in the lab; the defining test is never an input.
- Low kidney filtering: eGFR < 60 (CKD-EPI 2021), age 20+; inputs exclude creatinine.
- Protein in urine: urine albumin/creatinine >= 30 mg/g, age 20+; no urine test as input.
- Hepatitis C: HCV RNA positive, age 18+; inputs age, sex, ALT, AST, GGT, platelets, albumin, smoking.
- Train NHANES 2017-18; 20% stratified validation; logistic unless boosting wins by 0.02; threshold = highest with validation
  sensitivity >= 0.85. **Outside test, scored once:** NHANES 2021-23 (skipped if the cycle lacks the test or has < 20 cases).
  Verified if sensitivity and AUC are both >= 0.80.

In [ ]:
import io, json, urllib.request
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
YEAR = {"J": 2017, "L": 2021}


def nh(name, c):
    url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{YEAR[c]}/DataFiles/{name}_{c}.xpt"
    try:
        return pd.read_sas(io.BytesIO(urllib.request.urlopen(url, timeout=180).read()), format="xport")
    except Exception as e:
        print("missing", name, c, e); return pd.DataFrame({"SEQN": []})


def cycle(c):
    d = nh("DEMO", c)[["SEQN", "RIAGENDR", "RIDAGEYR"]]
    for f in ("BMX", "BIOPRO", "HDL", "GHB", "DIQ", "BPQ", "BPXO", "SMQ", "ALB_CR", "CBC", "HEPC"):
        x = nh(f, c); d = d.merge(x[["SEQN"] + [k for k in x.columns if k != "SEQN" and k not in d.columns]], on="SEQN", how="left")
    col = lambda k: d[k] if k in d else pd.Series(np.nan, index=d.index)
    told = col("DIQ010").map({1.0: 1.0, 2.0: 0.0, 3.0: 0.0})
    f = pd.DataFrame({"age": d.RIDAGEYR, "male": d.RIAGENDR.map({1.0: 1.0, 2.0: 0.0}), "bmi": col("BMXBMI"), "waist_cm": col("BMXWAIST"),
                      "triglycerides": col("LBXSTR"), "hdl": col("LBDHDD"), "alt": col("LBXSATSI"), "ggt": col("LBXSGTSI"), "uric_acid": col("LBXSUA"),
                      "systolic_bp": d[[k for k in ("BPXOSY1", "BPXOSY2", "BPXOSY3") if k in d]].mean(axis=1),
                      "high_bp": col("BPQ020").map({1.0: 1.0, 2.0: 0.0}),
                      "smoker_current": np.where(col("SMQ020") == 2, 0.0, col("SMQ040").map({1.0: 1.0, 2.0: 1.0, 3.0: 0.0}))})
    f["bun"] = col("LBXSBU")
    f["diabetes"] = told
    f["y_prediabetes"] = np.where(col("LBXGH").notna() & (told == 0) & (d.RIDAGEYR >= 18), (col("LBXGH") >= 5.7).astype(float), np.nan)
    cr, age, fem = col("LBXSCR"), d.RIDAGEYR, (d.RIAGENDR == 2).astype(int)
    k = np.where(fem == 1, 0.7, 0.9); a = np.where(fem == 1, -0.241, -0.302); r = cr / k
    egfr = 142 * np.minimum(r, 1) ** a * np.maximum(r, 1) ** -1.200 * 0.9938 ** age * np.where(fem == 1, 1.012, 1.0)
    known = cr.notna() & col("URDACT").notna() & (age >= 20)
    f["y_kidney"] = np.where(known, ((egfr < 60) | (col("URDACT") >= 30)).astype(float), np.nan)
    f["ast"], f["platelets"], f["albumin"] = col("LBXSASSI"), col("LBXPLTSI"), col("LBXSAL")
    f["y_low_egfr"] = np.where(cr.notna() & (age >= 20), (egfr < 60).astype(float), np.nan)
    f["y_albuminuria"] = np.where(col("URDACT").notna() & (age >= 20), (col("URDACT") >= 30).astype(float), np.nan)
    hcv = col("LBXHCR")  # hepatitis C RNA: 1 positive, 2 negative, 3 antibody negative (not tested for RNA)
    f["y_hepatitis_c"] = np.where(hcv.isin([1, 2, 3]) & (age >= 18), (hcv == 1).astype(float), np.nan)
    return f


def report(y, p, t):
    fl = p >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(fl[y == 1].mean()), 3),
         "specificity": round(float((~fl[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, p)), 3)}
    low = min(r["sensitivity"], r["auc"]); r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r


A, B = cycle("J"), cycle("L")
TASKS = {"low_egfr_labs": ("y_low_egfr", ["age", "male", "bmi", "systolic_bp", "high_bp", "diabetes", "uric_acid", "bun", "hdl"]),
         "albuminuria_labs": ("y_albuminuria", ["age", "male", "bmi", "systolic_bp", "high_bp", "diabetes", "uric_acid", "bun", "triglycerides", "hdl"]),
         "hepatitis_c_labs": ("y_hepatitis_c", ["age", "male", "alt", "ast", "ggt", "platelets", "albumin", "smoker_current"])}
for t in TASKS:
    print(t, "2017-18:", int(A[TASKS[t][0]].notna().sum()), int(A[TASKS[t][0]].sum()), "| 2021-23:", int(B[TASKS[t][0]].notna().sum()), int(B[TASKS[t][0]].sum()), flush=True)
out = {}
for target, (ycol, feats) in TASKS.items():
    a, b = A[A[ycol].notna()], B[B[ycol].notna()]
    if len(b) == 0 or b[ycol].sum() < 20:
        print('skipped (no outside data):', target); continue
    x, y = a[feats].to_numpy(np.float32), a[ycol].to_numpy(int)
    tr, va = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y)
    fits = {"logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=3000)).fit(x[tr], y[tr]),
            "boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=42).fit(x[tr], y[tr])}
    aucs = {k: roc_auc_score(y[va], m.predict_proba(x[va])[:, 1]) for k, m in fits.items()}
    name = "boosting" if aucs["boosting"] > aucs["logistic"] + 0.02 else "logistic"; model = fits[name]
    pv = model.predict_proba(x[va])[:, 1]
    thr = float(max(v for v in np.unique(np.round(pv, 5)) if (pv[y[va] == 1] >= v).mean() >= 0.85))
    res = report(b[ycol].to_numpy(int), model.predict_proba(b[feats].to_numpy(np.float32))[:, 1], thr)
    print(f"OUTSIDE TEST on NHANES 2021-23 (once): {target} {name} train {len(y)} ({int(y.sum())}+) val AUC {aucs[name]:.3f} -> {res}", flush=True)
    meta = {"target": target, "features": feats, "model": name, "threshold": thr, "min_age": 18 if target == "hepatitis_c_labs" else 20,
            "metrics": {"validation_auc": round(aucs[name], 3), "test": res}, "trained_on": f"US CDC NHANES 2017-18, {len(tr):,} people",
            "test_set": "NHANES 2021-23, different people", "median": np.nanmedian(x[tr], axis=0).tolist(),
            "train_min": np.nanmin(x[tr], axis=0).tolist(), "train_max": np.nanmax(x[tr], axis=0).tolist()}
    if name == "logistic":
        imp, sc, clf = (model.steps[i][1] for i in range(3))
        meta |= {"impute": imp.statistics_.tolist(), "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(), "coef": clf.coef_[0].tolist(), "intercept": float(clf.intercept_[0])}
    else:
        meta |= {"trees": [{k: p[0].nodes[k].tolist() for k in ("feature_idx", "num_threshold", "left", "right", "is_leaf", "value", "missing_go_to_left")} for p in model._predictors],
                 "baseline": float(np.ravel(model._baseline_prediction)[0])}
    json.dump(meta, open(f"/kaggle/working/{target}.json", "w"))
    out[target] = res
json.dump(out, open("/kaggle/working/summary.json", "w"), indent=1)